# 05 — Hoisting the shared per-patch work

**Did each implementation factor the patch-level climate work out of the
per-organism loop?**

In the ForeverTree spec `%T` and `%P` depend only on (patch, timestep), so they
are identical for all 10 trees on a patch — only the stochastic term varies.
The available optimisation is to compute that shared work once per patch-step.
The expert-authored references do exactly this (Josh: a patch-scope
`conditionImpact` the trees read via `here.`; Mesa: a `(cell, timestep)` cache
under the climate accessor); the AI-authored Mesa reference does not.

Source: per-cell `scorer.hoist.json` (schema `hoist-v2`), written by
`pixi run hoist-judge` — rubric in [`prompts/HOIST_JUDGE.md`](../prompts/HOIST_JUDGE.md).
Answers are `yes` / `partial` (partial attempt — typically the climate
*lookup* is shared but the impact *math* is still per-organism) / `no` /
`n-a` (no simulation loop to judge).


In [ ]:
import json, os, re, csv
from collections import Counter, defaultdict
from pathlib import Path

# cwd-robust: notebook runs from analysis/ (pixi run lab) or repo root.
ROOT = Path.cwd() if (Path.cwd() / 'runs').is_dir() else Path.cwd().parent
AGG  = ROOT / 'analysis' / 'aggregated.csv'

# Same scope as 01_analysis.ipynb / 03_manuscript_claims.ipynb.
MODEL_ORDER  = ['sonnet','gemma','kimi','minimax','mistral','glm','qwen','nemotron','deepseek']
TARGET_ORDER = ['mesa','josh','josh-mcp']
BATCH_FILTER = ['headline-stage1-20260601','headline-rerep-202606020525',
                'headline-rerep-202606021512','fill-*']
_PAT = re.compile('^(' + '|'.join(b.replace('*','.*') for b in BATCH_FILTER) + ')$')
ANSWERS = ['yes','partial','no','n-a']

# aggregated.csv supplies the canonical model/target/outcome labels; the hoist
# records supply the answer. Join on run_id.
agg = {r['run_id']: r for r in csv.DictReader(AGG.open())
       if _PAT.match(r['batch_tag']) and r['model'] in MODEL_ORDER
       and r['target'] in TARGET_ORDER}

rows, errors = [], []
for path in sorted(ROOT.glob('runs/*/*/workspace/results/scorer.hoist.json')):
    run_id = path.parents[2].name
    meta = agg.get(run_id)
    if meta is None:
        continue  # out of the headline scope
    rec = json.loads(path.read_text())
    if rec.get('hoist') is None:
        errors.append((run_id, rec.get('parse_error','?')))
        continue
    rows.append({'run_id': run_id, 'model': meta['model'], 'target': meta['target'],
                 'judge': rec['judge_model_id'], **rec['hoist']})

judged = {r['run_id'] for r in rows}
print(f'{len(rows)} judged / {len(agg)} cells in scope'
      f'{"" if len(rows)==len(agg) else f"  ({len(agg)-len(judged)-len(errors)} not yet judged)"}')
if errors:
    print(f'{len(errors)} parse error(s):')
    for run_id, err in errors[:10]:
        print(f'   {run_id}: {err[:90]}')
print('judge:', ', '.join(sorted({r['judge'] for r in rows})) or 'n/a')

## Headline


In [ ]:
def tally(subset):
    c = Counter(r['answer'] for r in subset)
    return [c.get(a, 0) for a in ANSWERS]

def pct(n, d):
    return f'{n/d:5.0%}' if d else '    -'

by_target = {t: [r for r in rows if r['target'] == t] for t in TARGET_ORDER}

hdr = f"{'target':10s} {'n':>4s} " + ' '.join(f'{a:>12s}' for a in ANSWERS)
print(hdr); print('-' * len(hdr))
for t in TARGET_ORDER + ['ALL']:
    subset = rows if t == 'ALL' else by_target[t]
    counts = tally(subset)
    n = len(subset)
    cells_ = ' '.join(f'{c:>4d} {pct(c, n)}' for c in counts)
    print(f'{t:10s} {n:>4d} {cells_}')

# 'attempted' = yes + partial: the judge saw a deliberate move to share the
# work across organisms, whether or not it was carried all the way.
print()
for t in TARGET_ORDER:
    sub = [r for r in by_target[t] if r['answer'] != 'n-a']
    att = sum(1 for r in sub if r['answer'] in ('yes','partial'))
    full = sum(1 for r in sub if r['answer'] == 'yes')
    print(f'{t:10s} attempted {att:>3d}/{len(sub):<3d} {pct(att,len(sub))}'
          f'   fully hoisted {full:>3d}/{len(sub):<3d} {pct(full,len(sub))}')

In [ ]:
import matplotlib.pyplot as plt

# Ordinal ramp (no -> partial -> yes) in one hue, light->dark, plus a
# neutral for the off-scale n-a. Steps are the validated ordinal band for a
# light surface; nothing lighter reads against the panel.
COLORS  = {'no': '#86b6ef', 'partial': '#3987e5', 'yes': '#184f95', 'n-a': '#9a9a97'}
SURFACE = '#fcfcfb'
INK, MUTED = '#0b0b0b', '#52514e'

fig, ax = plt.subplots(figsize=(7.2, 2.2), dpi=150)
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)

ys = range(len(TARGET_ORDER))
for y, t in zip(ys, TARGET_ORDER):
    n = len(by_target[t]) or 1
    left = 0.0
    for a, c in zip(ANSWERS, tally(by_target[t])):
        if not c:
            continue
        w = c / n
        # 2px surface gap between segments, drawn as an edge in the surface colour.
        ax.barh(y, w, left=left, height=0.55, color=COLORS[a],
                edgecolor=SURFACE, linewidth=2)
        if w > 0.09:  # selective direct labels — only where the count fits
            ax.text(left + w/2, y, str(c), ha='center', va='center', fontsize=8,
                    color='#ffffff' if a in ('yes','partial') else INK)
        left += w

ax.set_yticks(list(ys)); ax.set_yticklabels(TARGET_ORDER, fontsize=9, color=INK)
ax.invert_yaxis()
ax.set_xlim(0, 1); ax.set_xticks([0, .25, .5, .75, 1])
ax.set_xticklabels(['0', '25%', '50%', '75%', '100%'], fontsize=8, color=MUTED)
ax.set_xlabel('share of cells', fontsize=8, color=MUTED)
ax.set_title('Shared per-patch work factored out, by target', fontsize=10,
             color=INK, loc='left', pad=10)
for s in ax.spines.values():
    s.set_visible(False)
ax.tick_params(length=0)
handles = [plt.Rectangle((0,0),1,1, color=COLORS[a]) for a in ANSWERS]
ax.legend(handles, ANSWERS, ncol=4, fontsize=8, frameon=False,
          loc='upper left', bbox_to_anchor=(0, -0.35), labelcolor=MUTED)
plt.tight_layout()
OUT = ROOT / 'analysis' / 'figures'; OUT.mkdir(exist_ok=True)
fig.savefig(OUT / 'hoist_by_target.png', dpi=300, bbox_inches='tight',
            facecolor=SURFACE)
plt.show()

## By model × target

`Y` = yes, `p` = partial, `.` = no, `-` = n-a, blank = not judged.


In [ ]:
GLYPH = {'yes':'Y', 'partial':'p', 'no':'.', 'n-a':'-'}
cellmap = defaultdict(list)
for r in rows:
    cellmap[(r['model'], r['target'])].append(GLYPH[r['answer']])

w = max(len(m) for m in MODEL_ORDER)
print(f"{'':{w}s}  " + '  '.join(f'{t:<10s}' for t in TARGET_ORDER))
for m in MODEL_ORDER:
    line = f'{m:{w}s}  '
    for t in TARGET_ORDER:
        line += f"{''.join(sorted(cellmap[(m, t)], key='Yp.-'.index)):<10s}  "
    print(line)

## What the hoisting looked like

Mechanisms the judge named, for the cells that did something.


In [ ]:
for answer in ('yes', 'partial'):
    sub = [r for r in rows if r['answer'] == answer]
    print(f'=== {answer} ({len(sub)}) ' + '=' * 40)
    for mech, n in Counter(r['mechanism'].lower() for r in sub).most_common():
        print(f'  {n:>3d}  {mech}')
    print()

In [ ]:
# Full justifications, for reading the calls that matter.
for r in sorted(rows, key=lambda r: (ANSWERS.index(r['answer']), r['run_id'])):
    if r['answer'] in ('no', 'n-a'):
        continue
    print(f"{r['answer']:<11s} {r['run_id']}")
    print(f"            residual: {r['residual']}")
    print(f"            {r['evidence']}")
    print(f"            {r['justification']}\n")

## Does hoisting track with a working implementation?

Cross-tab against the headline pass definition (`substantive_conformance` AND
`did_run`) from `aggregated.csv` — a check that the hoist answer is measuring
code structure and not just picking out the cells that produced anything at all.


In [ ]:
def _bool(x): return str(x).strip().lower() == 'true'

xt = Counter()
for r in rows:
    a = agg[r['run_id']]
    passed = _bool(a['substantive_conformance']) and _bool(a['did_run'])
    xt[(r['answer'], 'pass' if passed else 'fail')] += 1

print(f"{'answer':<12s} {'pass':>6s} {'fail':>6s}")
for a in ANSWERS:
    p, f = xt[(a, 'pass')], xt[(a, 'fail')]
    if p or f:
        print(f'{a:<12s} {p:>6d} {f:>6d}')